# Task 2 - Original eLCS Baseline on Raw Dataset

This notebook runs the original scikit-eLCS implementation on the raw/minimally processed Phase I dataset.

Only processing required for a valid eLCS experiment is performed. Data-quality improvements and feature engineering are reserved for Task 3.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

from skeLCS import eLCS

In [2]:
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    auc
)

In [3]:
# Load the original dataset before any Task 3 cleaning
data = pd.read_csv("../data/patient_data.csv")

data.head()

,composite_key,age_level,gender,bmi_level,smoking,diabetes,age,age_normalized,bmi,hypertension,...,salt_intake,heart_rate,hdl,ldl,education_level,employment_status,source_dataset,disease_flags,sublabel,label
0,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,9,0.080460,19.20,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
1,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,22.55,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
2,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,22.89,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
3,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,23.12,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
4,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,4,0.022989,19.61,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal


In [4]:
# Confirm that this really is the raw dataset
print("Dataset shape:", data.shape)
print("Duplicate rows:", data.duplicated().sum())
print("Missing values:", data.isna().sum().sum())

Dataset shape: (280985, 39)
Duplicate rows: 2284
Missing values: 0


## 2.3 Raw Dataset and Target Creation

The original dataset is retained for the baseline experiment. The binary target is created from `sublabel`, where records containing `HT` represent heart disease.

In [5]:
classLabel = "heart_disease_target"

# HT in sublabel represents a record containing heart disease
data[classLabel] = (
    data["sublabel"]
    .str.contains("HT", na=False)
    .astype(int)
)

print("Target counts:")
print(data[classLabel].value_counts())

print("\nTarget proportions:")
print(data[classLabel].value_counts(normalize=True))

Target counts:
heart_disease_target
0    275081
1      5904
Name: count, dtype: int64

Target proportions:
heart_disease_target
0    0.978988
1    0.021012
Name: proportion, dtype: float64


In [6]:
# Check that every HT-containing sublabel maps to class 1
target_check = pd.crosstab(
    data["sublabel"],
    data[classLabel]
)

target_check

heart_disease_target,0,1
sublabel,,
DI,31918,0
DI_HT,0,1398
DI_HT_HY,0,845
DI_HY,66473,0
HT,0,2595
HT_HY,0,1066
HY,69806,0
N,106884,0


## 2.4 Predictor and Target Separation

Columns containing direct or partial disease-label information are excluded to prevent target leakage. No data-quality cleaning, duplicate removal, feature selection, or feature engineering is performed at this stage.

In [7]:
# These columns reveal direct or partial information about the target
leakage_columns = [
    "sublabel",
    "disease_flags",
    "label",
    "heart_disease"
]

X_raw = data.drop(
    columns=leakage_columns + [classLabel]
)

y = data[classLabel].to_numpy()

headers = X_raw.columns.to_numpy()

print("Predictor shape:", X_raw.shape)
print("Target shape:", y.shape)

print("\nPredictor columns:")
print(X_raw.columns.tolist())

Predictor shape: (280985, 35)
Target shape: (280985,)

Predictor columns:
['composite_key', 'age_level', 'gender', 'bmi_level', 'smoking', 'diabetes', 'age', 'age_normalized', 'bmi', 'hypertension', 'HbA1c_level', 'glucose', 'cholesterol', 'sleep_hours', 'triglycerides', 'physical_activity', 'family_history', 'stress_level', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'blood_pressure', 'high_blood_pressure', 'sugar_consumption', 'crp_level', 'homocysteine_level', 'systolic_bp', 'diastolic_bp', 'alcohol_intake', 'salt_intake', 'heart_rate', 'hdl', 'ldl', 'education_level', 'employment_status', 'source_dataset']


In [8]:
print("Target classes:", np.unique(y))

categorical_columns = X_raw.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numerical_columns = X_raw.select_dtypes(
    exclude=["object", "string", "category"]
).columns.tolist()

print("\nCategorical columns:", len(categorical_columns))
print(categorical_columns)

print("\nNumerical columns:", len(numerical_columns))
print(numerical_columns)

Target classes: [0 1]

Categorical columns: 16
['composite_key', 'age_level', 'gender', 'bmi_level', 'smoking', 'diabetes', 'physical_activity', 'family_history', 'stress_level', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'high_blood_pressure', 'sugar_consumption', 'education_level', 'employment_status', 'source_dataset']

Numerical columns: 19
['age', 'age_normalized', 'bmi', 'hypertension', 'HbA1c_level', 'glucose', 'cholesterol', 'sleep_hours', 'triglycerides', 'blood_pressure', 'crp_level', 'homocysteine_level', 'systolic_bp', 'diastolic_bp', 'alcohol_intake', 'salt_intake', 'heart_rate', 'hdl', 'ldl']


## 2.5 Train-Test Split

An 80/20 stratified train-test split is used. Stratification preserves the approximately 2.1% heart-disease rate in both sets.

In [9]:
SEED = 3

# Keep 20% unseen for evaluation and preserve the class distribution
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
    stratify=y
)

print("Training shape:", X_train_df.shape)
print("Testing shape:", X_test_df.shape)

print("\nTraining positive rate:", y_train.mean())
print("Testing positive rate:", y_test.mean())

Training shape: (224788, 35)
Testing shape: (56197, 35)

Training positive rate: 0.021010908055590156
Testing positive rate: 0.021015356691638343


## 2.6 Minimal eLCS Compatibility Processing

scikit-eLCS requires numeric NumPy arrays. Therefore, categorical string attributes are converted to numeric codes. The encoder is fitted only on the training data and then applied to the test data.

This is compatibility processing only. No scaling, discretisation, duplicate removal, feature engineering, or class-imbalance treatment is performed.

In [10]:
# eLCS requires every input value to be numeric
categorical_columns = X_train_df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train_encoded = X_train_df.copy()
X_test_encoded = X_test_df.copy()

# Learn category mappings from training data only
X_train_encoded[categorical_columns] = encoder.fit_transform(
    X_train_df[categorical_columns]
)

# Apply the same mappings to unseen test data
X_test_encoded[categorical_columns] = encoder.transform(
    X_test_df[categorical_columns]
)

X_train = X_train_encoded.to_numpy(dtype=float)
X_test = X_test_encoded.to_numpy(dtype=float)

y_train = y_train.astype(int)
y_test = y_test.astype(int)

print("X_train type:", type(X_train))
print("X_test type:", type(X_test))

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("All training features numeric:", np.issubdtype(X_train.dtype, np.number))
print("All testing features numeric:", np.issubdtype(X_test.dtype, np.number))

X_train type: <class 'numpy.ndarray'>
X_test type: <class 'numpy.ndarray'>
X_train shape: (224788, 35)
X_test shape: (56197, 35)
All training features numeric: True
All testing features numeric: True


## 2.7 Original eLCS Baseline

The supplied eLCS implementation is used without modifying its algorithm. The default eLCS hyperparameters are retained, apart from the random seed and explicit identification of discrete attributes.

In [11]:
# Create the original eLCS model using its default parameters
baseline_model = eLCS(
    random_state=SEED
)

print("Original eLCS baseline created successfully.")

Original eLCS baseline created successfully.


In [12]:
parameters = baseline_model.get_params()

parameter_table = pd.DataFrame({
    "Parameter": parameters.keys(),
    "Value": [
        value.tolist() if isinstance(value, np.ndarray) else value
        for value in parameters.values()
    ]
})

parameter_table

,Parameter,Value
0,N,1000
1,acc_sub,0.99
2,beta,0.2
3,chi,0.8
4,delta,0.1
5,discrete_attribute_limit,10
6,do_GA_subsumption,True
7,do_correct_set_subsumption,False
8,fitness_reduction,0.1
9,init_fit,0.01


In [13]:
start_time = time.perf_counter()

trained_model = baseline_model.fit(
    X_train,
    y_train
)

training_time = time.perf_counter() - start_time

print(f"Training completed in {training_time:.2f} seconds")

Training completed in 43.31 seconds


In [14]:
predictions = trained_model.predict(X_test)

print("Number of predictions:", len(predictions))
print("Unique predictions:", np.unique(predictions))

unmatched_predictions = pd.isna(predictions).sum()

print("Unmatched predictions:", unmatched_predictions)

Number of predictions: 56197
Unique predictions: [0]
Unmatched predictions: 0


In [15]:
# Make class predictions and probability predictions
predictions = trained_model.predict(X_test)
probabilities = trained_model.predict_proba(X_test)

# Probability assigned to class 1 (heart disease)
positive_probabilities = probabilities[:, 1]

# Standard classification metrics
accuracy = accuracy_score(
    y_test,
    predictions
)

precision = precision_score(
    y_test,
    predictions,
    zero_division=0
)

recall = recall_score(
    y_test,
    predictions,
    zero_division=0
)

f1 = f1_score(
    y_test,
    predictions,
    zero_division=0
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    predictions
)

# eLCS score() reports balanced accuracy
elcs_score = trained_model.score(
    X_test,
    y_test
)

# ROC-AUC uses probability scores, not the final class predictions
fpr, tpr, _ = roc_curve(
    y_test,
    positive_probabilities
)

roc_auc = auc(
    fpr,
    tpr
)

# PR-AUC also uses probability scores
precision_curve, recall_curve, _ = precision_recall_curve(
    y_test,
    positive_probabilities
)

pr_auc = auc(
    recall_curve,
    precision_curve
)

results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Balanced Accuracy",
        "eLCS score()",
        "ROC-AUC",
        "PR-AUC",
        "Training Time (seconds)"
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        f1,
        balanced_accuracy,
        elcs_score,
        roc_auc,
        pr_auc,
        training_time
    ]
})

results

,Metric,Value
0,Accuracy,0.978985
1,Precision,0.000000
2,Recall,0.000000
3,F1,0.000000
4,Balanced Accuracy,0.500000
5,eLCS score(),0.500000
6,ROC-AUC,0.713849
7,PR-AUC,0.107773
8,Training Time (seconds),43.306305


In [17]:
cm = confusion_matrix(
    y_test,
    predictions,
    labels=[0, 1]
)

confusion_matrix_table = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

confusion_matrix_table

,Predicted 0,Predicted 1
Actual 0,55016,0
Actual 1,1181,0


In [18]:
tn, fp, fn, tp = cm.ravel()

print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

True negatives: 55016
False positives: 0
False negatives: 1181
True positives: 0


In [ ]:
training_balanced_accuracy = trained_model.get_final_accuracy()
training_coverage = trained_model.get_final_instance_coverage()

print(
    "Final training balanced accuracy:",
    training_balanced_accuracy
)

print(
    "Final training instance coverage:",
    training_coverage
)

Final training balanced accuracy: 0.5
Final training instance coverage: 1.0


In [19]:
single_run_summary = pd.DataFrame({
    "Seed": [SEED],
    "Train Rows": [len(y_train)],
    "Test Rows": [len(y_test)],
    "Test Positive Rate": [y_test.mean()],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1": [f1],
    "Balanced Accuracy": [balanced_accuracy],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc],
    "TN": [tn],
    "FP": [fp],
    "FN": [fn],
    "TP": [tp],
    "Training Time": [training_time]
})

single_run_summary

,Seed,Train Rows,Test Rows,Test Positive Rate,Accuracy,Precision,Recall,F1,Balanced Accuracy,ROC-AUC,PR-AUC,TN,FP,FN,TP,Training Time
0,3,224788,56197,0.021015,0.978985,0.0,0.0,0.0,0.5,0.713849,0.107773,55016,0,1181,0,43.306305


In [ ]:
seeds = [
    3, 6, 9, 12, 15,
    18, 21, 24, 27, 30
]

In [20]:
def run_baseline(seed):
    # Create a reproducible stratified 80/20 split
    X_train_df, X_test_df, y_train, y_test = train_test_split(
        X_raw,
        y,
        test_size=0.20,
        random_state=seed,
        shuffle=True,
        stratify=y
    )

    # Convert categorical values to numeric codes required by eLCS
    categorical_columns = X_train_df.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    X_train_encoded = X_train_df.copy()
    X_test_encoded = X_test_df.copy()

    X_train_encoded[categorical_columns] = encoder.fit_transform(
        X_train_df[categorical_columns]
    )

    X_test_encoded[categorical_columns] = encoder.transform(
        X_test_df[categorical_columns]
    )

    X_train = X_train_encoded.to_numpy(dtype=float)
    X_test = X_test_encoded.to_numpy(dtype=float)

    # Tell eLCS which encoded attributes should be treated as discrete
    discrete_columns = []

    for column in X_train_df.columns:
        if (
            column in categorical_columns
            or X_train_df[column].nunique(dropna=True) <= 10
        ):
            discrete_columns.append(column)

    specified_discrete_attributes = np.array(
        [
            X_train_df.columns.get_loc(column)
            for column in discrete_columns
        ],
        dtype=int
    )

    # Keep the original eLCS configuration consistent across runs
    model = eLCS(
        learning_iterations=10000,
        discrete_attribute_limit="d",
        specified_attributes=specified_discrete_attributes,
        random_state=seed
    )

    start_time = time.perf_counter()

    model.fit(
        X_train,
        y_train
    )

    training_time = time.perf_counter() - start_time

    predictions = model.predict(X_test)

    unmatched = pd.isna(predictions).sum()

    if unmatched != 0:
        raise ValueError(
            f"Seed {seed} produced {unmatched} unmatched test predictions."
        )

    probabilities = model.predict_proba(X_test)
    positive_probabilities = probabilities[:, 1]

    # Standard classification metrics
    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        predictions
    )

    # ROC-AUC
    fpr, tpr, _ = roc_curve(
        y_test,
        positive_probabilities
    )

    roc_auc = auc(
        fpr,
        tpr
    )

    # PR-AUC
    precision_curve, recall_curve, _ = precision_recall_curve(
        y_test,
        positive_probabilities
    )

    pr_auc = auc(
        recall_curve,
        precision_curve
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Seed": seed,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Balanced Accuracy": balanced_accuracy,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Training Time": training_time
    }

In [21]:
seeds = [
    3, 6, 9, 12, 15,
    18, 21, 24, 27, 30
]

baseline_results = []

for seed in seeds:
    print(f"Running baseline with seed {seed}...")

    result = run_baseline(seed)
    baseline_results.append(result)

baseline_results = pd.DataFrame(
    baseline_results
)

baseline_results

Running baseline with seed 3...
Running baseline with seed 6...
Running baseline with seed 9...
Running baseline with seed 12...
Running baseline with seed 15...
Running baseline with seed 18...
Running baseline with seed 21...
Running baseline with seed 24...
Running baseline with seed 27...
Running baseline with seed 30...


,Seed,Accuracy,Precision,Recall,F1,Balanced Accuracy,ROC-AUC,PR-AUC,TN,FP,FN,TP,Training Time
0,3,0.978985,0.000000,0.000000,0.000000,0.500000,0.603521,0.051872,55016,0,1181,0,40.803142
1,6,0.978985,0.000000,0.000000,0.000000,0.500000,0.610371,0.062621,55016,0,1181,0,38.277559
2,9,0.978985,0.000000,0.000000,0.000000,0.500000,0.568046,0.078381,55016,0,1181,0,38.114931
3,12,0.978985,0.000000,0.000000,0.000000,0.500000,0.572398,0.061368,55016,0,1181,0,41.825586
4,15,0.978469,0.177778,0.006774,0.013051,0.503051,0.585462,0.079467,54979,37,1173,8,38.210348
5,18,0.978985,0.000000,0.000000,0.000000,0.500000,0.585208,0.061329,55016,0,1181,0,38.074161
6,21,0.978985,0.000000,0.000000,0.000000,0.500000,0.676333,0.080623,55016,0,1181,0,36.169320
7,24,0.978985,0.000000,0.000000,0.000000,0.500000,0.604437,0.068572,55016,0,1181,0,1714.127018
8,27,0.978985,0.000000,0.000000,0.000000,0.500000,0.698213,0.108668,55016,0,1181,0,110.106360
9,30,0.978949,0.250000,0.000847,0.001688,0.500396,0.602275,0.062603,55013,3,1180,1,111.822284


In [22]:
metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced Accuracy",
    "ROC-AUC",
    "PR-AUC",
    "Training Time"
]

baseline_summary = baseline_results[
    metric_columns
].agg(
    ["mean", "std"]
).T

baseline_summary

,mean,std
Accuracy,0.978929,0.000162
Precision,0.042778,0.091776
Recall,0.000762,0.002129
F1,0.001474,0.004102
Balanced Accuracy,0.500345,0.000959
ROC-AUC,0.610627,0.043078
PR-AUC,0.071550,0.016102
Training Time,220.753071,525.577986


In [23]:
confusion_summary = baseline_results[
    ["TN", "FP", "FN", "TP"]
].agg(
    ["mean", "std"]
).T

confusion_summary

,mean,std
TN,55012.0,11.633286
FP,4.0,11.633286
FN,1180.1,2.514403
TP,0.9,2.514403


In [24]:
baseline_results.to_csv(
    "Task_2_Baseline_Results.csv",
    index=False
)

print("Baseline results saved.")

Baseline results saved.
